# ПР-10. Кластеризация и сегментация

**Цель:** применить K-means для сегментации наблюдений и описать полученные группы. Используются признаки `total_bill`, `tip` и `size` из набора `tips_dirty.csv`.

**Важно:** у кластеризации без целевой разметки нет объективной проверки правильности групп. Метрики ниже помогают сравнить варианты, но сами по себе не доказывают, что сегменты полезны.

## 1. Загрузка и обзор данных

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ROOT=Path.cwd()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
df=pd.read_csv(ROOT/'data'/'tips_dirty.csv')
df.head(), df.isna().sum()

## 2. Масштабирование и выбор числа кластеров k

Признаки измеряются в разных единицах, поэтому перед K-means стандартизируем их. Сравним инерцию (elbow) и silhouette для нескольких значений k.

In [ ]:
features=['total_bill','tip','size']
work=df[features].dropna().copy()
scaler=StandardScaler(); Z=scaler.fit_transform(work)
ks=range(2,8)
inertias=[]; silhouettes=[]
for k in ks:
    km=KMeans(n_clusters=k,random_state=42,n_init=20).fit(Z)
    inertias.append(km.inertia_); silhouettes.append(silhouette_score(Z,km.labels_))
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].plot(list(ks),inertias,marker='o'); ax[0].set_title('Elbow')
ax[1].plot(list(ks),silhouettes,marker='o'); ax[1].set_title('Silhouette')
plt.show()

## 3. Выбор числа кластеров

Выберем k, сочетая elbow-график, silhouette и возможность содержательно интерпретировать сегменты. Максимум silhouette — ориентир, а не единственный критерий.

In [ ]:
chosen_k=3  # баланс читаемой сегментации и качества по silhouette
km=KMeans(n_clusters=chosen_k,random_state=42,n_init=20)
work['cluster']=km.fit_predict(Z)
summary=work.groupby('cluster')[features].agg(['count','mean','median']).round(2)
display(summary)
print('silhouette:',round(silhouette_score(Z,work['cluster']),3))
print('Размеры кластеров:')
display(work['cluster'].value_counts().sort_index().rename('count').to_frame())

## 4. Интерпретация результатов

На графиках инерция убывает с ростом k; наиболее заметное уплощение наблюдается после k=3. Silhouette максимален при k=3 (0,300; для k=2 — 0,282), поэтому для компактной сегментации выбран вариант из трёх групп. Значение silhouette умеренное: кластеры разделены не идеально.

При k=3 получены следующие профили (средние и медианы):

- **Кластер 0 (114 наблюдений):** средний счёт 23,70 (медиана 24,51), чаевые 3,82 (3,94), размер компании 2,01 (2). Небольшие компании со средним чеком и чаевыми около общего уровня.
- **Кластер 1 (92 наблюдения):** средний счёт 23,13 (22,67), чаевые 3,68 (3,68), размер компании 4,95 (5). Более крупные компании при сопоставимом среднем чеке и чаевых.
- **Кластер 2 (4 наблюдения):** средний счёт 86,20 (78,53), чаевые 4,30 (4,68), размер компании 3,00 (3). Редкие наблюдения с исключительно высоким счётом.

### Ответы на вопросы

1. Стандартизация нужна, чтобы признаки с большим числовым масштабом не доминировали при вычислении расстояний K-means.
2. Выбрано k=3: при нём лучший silhouette среди проверенных вариантов, а elbow-график показывает убывающую отдачу от дальнейшего дробления.
3. Основное различие кластеров 0 и 1 — размер компании (около 2 против 5 человек); кластер 2 отделяет редкие очень большие счета.
4. Silhouette 0,300 указывает на умеренную, нечеткую структуру.

**Ограничение:** без целевой разметки кластеризация не имеет объективной проверки правильности. Метрики оценивают геометрию групп, а полезность сегментов нужно подтверждать предметной задачей и внешними данными.